# release16_rulebase_reversal_drawdown_upside_target_v1

Train-only target revision after `release16_rulebase_reversal_failed_fold_diagnostic_v1`.

Design:
- `wf_2020`: cap high-drawdown rulebase-reversal labels by projecting them into downtrend/risk-control target states.
- `wf_2023_2024`: add rebound/upside participation labels from train-window forward-return states while preserving the drawdown guard.
- `wf_2022`: leave guard-fold target rows unchanged.

This notebook writes a target contract only. It does not promote or retain an actor.


In [1]:

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import sys
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'SPEC.md').exists():
            return candidate
    raise RuntimeError(f'repo root not found from {start}')

ROOT = find_repo_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'release16_rulebase_reversal_drawdown_upside_target_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD_NAME = 'release16_rulebase_reversal_checkpoint_readiness_v1'
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / SOURCE_CARD_NAME
SOURCE_CONTRACT = SOURCE_CARD / 'rulebase_reversal_bc_target_contract.csv'
SOURCE_SUMMARY = SOURCE_CARD / 'summary.json'
SOURCE_DIAGNOSTIC = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_rulebase_reversal_failed_fold_diagnostic_v1' / 'summary.json'
REFERENCE_BASELINE = 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02'
DIRECT_COMPARISON_BASELINE = 'release16_rulebase_reversal_checkpoint_metric_replay_v1'
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}
FAILED_FOLDS = ('wf_2020', 'wf_2023_2024')
GUARD_FOLDS = ('wf_2022',)

RECIPE = {
    'recipe_id': 'release16_rulebase_reversal_drawdown_upside_target_v1',
    'wf_2020_drawdown_cap': {
        'fold': 'wf_2020_train',
        'trigger': 'rulebase_reversal_overlay_state and strategy_drawdown_prev_overlay >= 0.08',
        'cap_position': 0.35,
        'reason': 'high_drawdown_rulebase_lift_cap',
    },
    'wf_2023_2024_upside_floor': {
        'fold': 'wf_2023_2024_train',
        'trigger': 'forward_return_5 >= 0.02 or forward_return_10 >= 0.05 or forward_return_21 >= 0.08, with non-downtrend and vol/drawdown guard',
        'floor_position': 0.55,
        'mom5_prev_min': -0.12,
        'dd63_prev_min': -0.60,
        'rv21_max': 2.80,
        'reason': 'train_only_upside_rebound_floor',
    },
    'wf_2022_guard': {'description': 'Preserve guard-fold target rows unchanged.'},
}

SIDE_CARS = {
    'target_contract': CARD / 'drawdown_upside_bc_target_contract.csv',
    'adjusted_rows': CARD / 'drawdown_upside_adjusted_rows.csv',
    'scope_audit': CARD / 'target_scope_audit.csv',
    'distribution': CARD / 'target_distribution_before_after.csv',
    'bucket_delta': CARD / 'target_bucket_delta.csv',
    'feature_contrast': CARD / 'target_feature_contrast.csv',
    'consumed_audit': CARD / 'consumed_target_audit.csv',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return rel(value) if ROOT in [value, *value.parents] else str(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def bool_series(frame: pd.DataFrame, col: str) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(False, index=frame.index)
    s = frame[col]
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0.0)
    return s.fillna('').astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y', 't'})


def numeric(frame: pd.DataFrame, col: str, default: float = np.nan) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(default, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[col], errors='coerce')


def consumed_target(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(
        [state_objective_target_position_from_row(row) for row in frame.to_dict('records')],
        index=frame.index,
        dtype='float64',
    )

print('card', rel(CARD))
print('source_contract', rel(SOURCE_CONTRACT))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


card runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_target_v1
source_contract runtime/model_selection/card/release16_rulebase_reversal_checkpoint_readiness_v1/rulebase_reversal_bc_target_contract.csv


In [2]:

source_summary = load_json(SOURCE_SUMMARY)
diagnostic_summary = load_json(SOURCE_DIAGNOSTIC)
source = pd.read_csv(SOURCE_CONTRACT, parse_dates=['datetime', 'train_start', 'train_end', 'eval_start', 'eval_end'])
required = {
    'datetime', 'fold', 'train_start', 'train_end', 'eval_start', 'eval_end',
    'teacher_position', 'risk_target_position', 'rebound_target_position',
    'objective_weight', 'is_downtrend_state', 'is_rebound_cluster', 'drawdown_control_state',
    'rulebase_reversal_overlay_state', 'strategy_drawdown_prev_overlay',
    'soxl_mom5_prev_overlay', 'soxl_dd63_prev_overlay', 'forward_return_5',
    'forward_return_10', 'forward_return_21', 'factor_rv21_overlay',
}
missing = sorted(required - set(source.columns))
if missing:
    raise ValueError(f'source contract missing columns: {missing}')

for col in ['is_downtrend_state', 'is_rebound_cluster', 'drawdown_control_state', 'release_gate_state', 'factor_rebound_state', 'rulebase_reversal_overlay_state']:
    source[col] = bool_series(source, col)
for col in [
    'teacher_position', 'risk_target_position', 'rebound_target_position', 'objective_weight',
    'strategy_drawdown_prev_overlay', 'soxl_mom5_prev_overlay', 'soxl_dd63_prev_overlay',
    'forward_return_5', 'forward_return_10', 'forward_return_21', 'factor_rv21_overlay',
]:
    source[col] = numeric(source, col)

source['source_consumed_target_position'] = consumed_target(source)
source['source_risk_target_position_before_drawdown_upside'] = source['risk_target_position']
source['source_rebound_target_position_before_drawdown_upside'] = source['rebound_target_position']
source['source_is_downtrend_state_before_drawdown_upside'] = source['is_downtrend_state']
source['source_is_rebound_cluster_before_drawdown_upside'] = source['is_rebound_cluster']
source['source_drawdown_control_state_before_drawdown_upside'] = source['drawdown_control_state']
print('source rows', len(source), 'folds', sorted(source['fold'].astype(str).unique()))
print('source decision', source_summary.get('decision'))
print('diagnostic decision', diagnostic_summary.get('decision'))
print('diagnostic next', diagnostic_summary.get('next_mechanism'))


source rows 3731 folds ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train']
source decision rulebase_reversal_training_contract_ready_for_checkpoint_replay_no_promotion
diagnostic decision rulebase_reversal_failed_fold_diagnostic_requires_target_revision_no_promotion
diagnostic next build_drawdown_capped_rulebase_reversal_and_wf2024_upside_participation_target_then_rerun_checkpoint_metric_replay


In [3]:

out = source.sort_values(['fold', 'datetime']).reset_index(drop=True).copy()

strict_train_masks = {
    fold: out['datetime'].between(pd.Timestamp(spec['train'][0]), pd.Timestamp(spec['train'][1]))
    for fold, spec in WALK_FORWARD_WINDOWS.items()
}
train_mask = pd.Series(False, index=out.index)
for mask in strict_train_masks.values():
    train_mask = train_mask | mask
out['drawdown_upside_adjusted_state'] = False
out['drawdown_upside_adjustment_reason'] = 'source_contract'
out['drawdown_upside_target_position'] = out['source_consumed_target_position'].clip(0.0, 1.0)
out['drawdown_upside_target_delta_vs_source_consumed'] = 0.0
out['pre_drawdown_upside_objective_weight'] = out['objective_weight'].fillna(1.0)

cap = RECIPE['wf_2020_drawdown_cap']
cap_mask = (
    strict_train_masks['wf_2020']
    & out['fold'].astype(str).eq(cap['fold'])
    & out['rulebase_reversal_overlay_state'].astype(bool)
    & out['strategy_drawdown_prev_overlay'].fillna(0.0).ge(float(cap['cap_position'] * 0 + 0.08))
    & out['source_consumed_target_position'].gt(float(cap['cap_position']) + 1e-12)
)
cap_target = float(cap['cap_position'])
out.loc[cap_mask, 'risk_target_position'] = cap_target
out.loc[cap_mask, 'rebound_target_position'] = np.minimum(pd.to_numeric(out.loc[cap_mask, 'rebound_target_position'], errors='coerce').fillna(cap_target), cap_target)
out.loc[cap_mask, 'is_downtrend_state'] = True
out.loc[cap_mask, 'drawdown_control_state'] = True
out.loc[cap_mask, 'is_rebound_cluster'] = False
out.loc[cap_mask, 'drawdown_upside_target_position'] = cap_target
out.loc[cap_mask, 'drawdown_upside_adjusted_state'] = True
out.loc[cap_mask, 'drawdown_upside_adjustment_reason'] = cap['reason']
out.loc[cap_mask, 'objective_weight'] = np.maximum(out.loc[cap_mask, 'objective_weight'].fillna(1.0), 10.0)

floor = RECIPE['wf_2023_2024_upside_floor']
upside_signal = (
    out['forward_return_5'].fillna(-9.0).ge(0.02)
    | out['forward_return_10'].fillna(-9.0).ge(0.05)
    | out['forward_return_21'].fillna(-9.0).ge(0.08)
)
safe_state = (
    ~out['is_downtrend_state'].astype(bool)
    & ~out['drawdown_control_state'].astype(bool)
    & out['soxl_mom5_prev_overlay'].fillna(0.0).ge(float(floor['mom5_prev_min']))
    & out['soxl_dd63_prev_overlay'].fillna(0.0).ge(float(floor['dd63_prev_min']))
    & out['factor_rv21_overlay'].fillna(0.0).le(float(floor['rv21_max']))
)
up_mask = (
    strict_train_masks['wf_2023_2024']
    & out['fold'].astype(str).eq(floor['fold'])
    & upside_signal
    & safe_state
    & out['source_consumed_target_position'].lt(float(floor['floor_position']) - 1e-12)
)
up_target = float(floor['floor_position'])
out.loc[up_mask, 'rebound_target_position'] = np.maximum(pd.to_numeric(out.loc[up_mask, 'rebound_target_position'], errors='coerce').fillna(0.0), up_target)
out.loc[up_mask, 'risk_target_position'] = np.maximum(pd.to_numeric(out.loc[up_mask, 'risk_target_position'], errors='coerce').fillna(0.0), up_target)
out.loc[up_mask, 'is_rebound_cluster'] = True
out.loc[up_mask, 'is_downtrend_state'] = False
out.loc[up_mask, 'drawdown_control_state'] = False
out.loc[up_mask, 'drawdown_upside_target_position'] = np.maximum(out.loc[up_mask, 'source_consumed_target_position'], up_target)
out.loc[up_mask, 'drawdown_upside_adjusted_state'] = True
out.loc[up_mask, 'drawdown_upside_adjustment_reason'] = floor['reason']
out.loc[up_mask, 'objective_weight'] = np.maximum(out.loc[up_mask, 'objective_weight'].fillna(1.0), 8.0)

out['drawdown_upside_consumed_target_position'] = consumed_target(out)
out['drawdown_upside_target_delta_vs_source_consumed'] = out['drawdown_upside_consumed_target_position'] - out['source_consumed_target_position']
out['drawdown_upside_target_mismatch'] = (out['drawdown_upside_consumed_target_position'] - out['drawdown_upside_target_position']).abs().gt(1e-9)
out['drawdown_upside_state_flag_changed'] = (
    out['is_downtrend_state'].ne(out['source_is_downtrend_state_before_drawdown_upside'])
    | out['is_rebound_cluster'].ne(out['source_is_rebound_cluster_before_drawdown_upside'])
    | out['drawdown_control_state'].ne(out['source_drawdown_control_state_before_drawdown_upside'])
)
out['target_semantics_revision'] = 'release16_rulebase_reversal_drawdown_upside_target_v1'
out['drawdown_upside_source_card'] = SOURCE_CARD_NAME
out['source_components'] = np.where(
    out['drawdown_upside_adjusted_state'],
    out.get('source_components', pd.Series('', index=out.index)).fillna('').astype(str) + ',release16_rulebase_reversal_drawdown_upside_target_v1',
    out.get('source_components', pd.Series('', index=out.index)).fillna('').astype(str),
)
if 'source_component_count' in out.columns:
    original_count = pd.to_numeric(out['source_component_count'], errors='coerce').fillna(0).astype(int)
    out['source_component_count'] = np.where(out['drawdown_upside_adjusted_state'], original_count + 1, original_count)

adjusted_rows = out[out['drawdown_upside_adjusted_state']].copy()
print('adjusted rows by fold')
print(adjusted_rows.groupby(['fold', 'drawdown_upside_adjustment_reason']).size().to_string())
display(adjusted_rows[['datetime','fold','drawdown_upside_adjustment_reason','source_consumed_target_position','drawdown_upside_consumed_target_position','strategy_drawdown_prev_overlay','forward_return_5','forward_return_10','forward_return_21']].head(20))


adjusted rows by fold
fold                drawdown_upside_adjustment_reason
wf_2020_train       high_drawdown_rulebase_lift_cap       92
wf_2023_2024_train  train_only_upside_rebound_floor      151


,datetime,fold,drawdown_upside_adjustment_reason,source_consumed_target_position,drawdown_upside_consumed_target_position,strategy_drawdown_prev_overlay,forward_return_5,forward_return_10,forward_return_21
282,2011-04-21,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.105640,NaN,NaN,NaN
283,2011-04-25,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.106911,NaN,NaN,NaN
284,2011-04-26,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.105560,NaN,NaN,NaN
285,2011-04-27,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.082993,NaN,NaN,NaN
286,2011-04-28,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.083072,NaN,NaN,NaN
287,2011-04-29,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.092846,NaN,NaN,NaN
288,2011-05-02,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.085774,NaN,NaN,NaN
295,2011-05-11,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.102858,NaN,NaN,NaN
297,2011-05-13,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.094753,NaN,NaN,NaN
300,2011-05-18,wf_2020_train,high_drawdown_rulebase_lift_cap,0.65,0.35,0.148868,NaN,NaN,NaN


In [4]:

def distribution(frame: pd.DataFrame, target_col: str, version: str) -> pd.DataFrame:
    rows = []
    for fold, group in frame.groupby('fold', sort=True):
        target = pd.to_numeric(group[target_col], errors='coerce')
        rows.append({
            'version': version,
            'fold': str(fold),
            'rows': int(len(group)),
            'mean_target_position': float(target.mean()),
            'median_target_position': float(target.median()),
            'share_ge_35pct': float(target.ge(0.35).mean()),
            'share_ge_50pct': float(target.ge(0.50).mean()),
            'share_ge_55pct': float(target.ge(0.55).mean()),
            'share_ge_65pct': float(target.ge(0.65).mean()),
            'rows_ge_35pct': int(target.ge(0.35).sum()),
            'rows_ge_50pct': int(target.ge(0.50).sum()),
            'rows_ge_55pct': int(target.ge(0.55).sum()),
            'rows_ge_65pct': int(target.ge(0.65).sum()),
            'downtrend_rows': int(bool_series(group, 'is_downtrend_state').sum()),
            'rebound_rows': int(bool_series(group, 'is_rebound_cluster').sum()),
            'drawdown_control_rows': int(bool_series(group, 'drawdown_control_state').sum()),
        })
    return pd.DataFrame(rows)

source_dist = distribution(out.assign(_target=out['source_consumed_target_position']), '_target', 'source_consumed')
revised_dist = distribution(out.assign(_target=out['drawdown_upside_consumed_target_position']), '_target', 'drawdown_upside')
target_distribution = pd.concat([source_dist, revised_dist], ignore_index=True)

bucket_edges = [-np.inf, 0.10, 0.30, 0.35, 0.50, 0.55, 0.65, np.inf]
bucket_labels = ['00_cash_lt10', '10_low_10_30', '30_cap_30_35', '35_mid_35_50', '50_upside_50_55', '55_high_55_65', '65_full_ge65']
bucket_frames = []
for version, col in [('source_consumed', 'source_consumed_target_position'), ('drawdown_upside', 'drawdown_upside_consumed_target_position')]:
    tmp = out[['datetime', 'fold']].copy()
    tmp['version'] = version
    tmp['bucket'] = pd.cut(pd.to_numeric(out[col], errors='coerce'), bins=bucket_edges, labels=bucket_labels, right=False).astype(str)
    bucket_frames.append(tmp)
bucket_counts = pd.concat(bucket_frames, ignore_index=True).groupby(['fold', 'bucket', 'version']).size().rename('rows').reset_index()
bucket_delta = bucket_counts.pivot_table(index=['fold', 'bucket'], columns='version', values='rows', fill_value=0).reset_index()
for col in ['source_consumed', 'drawdown_upside']:
    if col not in bucket_delta.columns:
        bucket_delta[col] = 0
bucket_delta['delta_rows'] = bucket_delta['drawdown_upside'].astype(int) - bucket_delta['source_consumed'].astype(int)

scope_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    target_fold = spec['target_fold']
    fold_rows = out[out['fold'].astype(str).eq(target_fold)].copy()
    train_start, train_end = pd.Timestamp(spec['train'][0]), pd.Timestamp(spec['train'][1])
    valid_start, valid_end = pd.Timestamp(spec['valid'][0]), pd.Timestamp(spec['valid'][1])
    test_start, test_end = pd.Timestamp(spec['test'][0]), pd.Timestamp(spec['test'][1])
    changed = fold_rows[fold_rows['drawdown_upside_adjusted_state']]
    scope_rows.append({
        'fold': fold_name,
        'target_fold': target_fold,
        'all_target_rows': int(len(fold_rows)),
        'train_target_rows': int(fold_rows['datetime'].between(train_start, train_end).sum()),
        'valid_overlap_rows': int(fold_rows['datetime'].between(valid_start, valid_end).sum()),
        'test_overlap_rows': int(fold_rows['datetime'].between(test_start, test_end).sum()),
        'changed_rows_all': int(len(changed)),
        'changed_rows_train': int(changed['datetime'].between(train_start, train_end).sum()) if not changed.empty else 0,
        'changed_rows_valid': int(changed['datetime'].between(valid_start, valid_end).sum()) if not changed.empty else 0,
        'changed_rows_test': int(changed['datetime'].between(test_start, test_end).sum()) if not changed.empty else 0,
        'target_contract_gate': bool(len(fold_rows) > 0 and fold_rows['datetime'].between(train_start, train_end).any() and not fold_rows['datetime'].between(test_start, test_end).any()),
    })
scope_audit = pd.DataFrame(scope_rows)

consumed_rows = []
for fold, group in out.groupby('fold', sort=True):
    changed = group['drawdown_upside_adjusted_state'].astype(bool)
    consumed_rows.append({
        'fold': str(fold),
        'rows': int(len(group)),
        'adjusted_rows': int(changed.sum()),
        'mismatch_rows': int(group.loc[changed, 'drawdown_upside_target_mismatch'].sum()) if changed.any() else 0,
        'state_flag_changed_rows': int(group['drawdown_upside_state_flag_changed'].sum()),
        'mean_source_consumed_target': float(group['source_consumed_target_position'].mean()),
        'mean_drawdown_upside_consumed_target': float(group['drawdown_upside_consumed_target_position'].mean()),
        'mean_delta_vs_source_consumed': float(group['drawdown_upside_target_delta_vs_source_consumed'].mean()),
        'sum_positive_delta': float(group['drawdown_upside_target_delta_vs_source_consumed'].clip(lower=0).sum()),
        'sum_negative_delta': float(group['drawdown_upside_target_delta_vs_source_consumed'].clip(upper=0).sum()),
    })
consumed_audit = pd.DataFrame(consumed_rows)

feature_cols = [
    'source_consumed_target_position', 'drawdown_upside_consumed_target_position', 'drawdown_upside_target_delta_vs_source_consumed',
    'strategy_drawdown_prev_overlay', 'soxl_mom5_prev_overlay', 'soxl_dd63_prev_overlay', 'factor_rv21_overlay',
    'forward_return_5', 'forward_return_10', 'forward_return_21', 'mom_5', 'mom_21', 'drawdown_63', 'rv_21',
]
contrast = []
for fold, group in out.groupby('fold', sort=True):
    changed = group['drawdown_upside_adjusted_state'].astype(bool)
    for feature in feature_cols:
        if feature not in group.columns:
            continue
        values = pd.to_numeric(group[feature], errors='coerce')
        changed_values = values[changed]
        unchanged_values = values[~changed]
        contrast.append({
            'fold': str(fold),
            'feature': feature,
            'changed_rows': int(changed.sum()),
            'unchanged_rows': int((~changed).sum()),
            'changed_mean': float(changed_values.mean()) if changed_values.notna().any() else np.nan,
            'unchanged_mean': float(unchanged_values.mean()) if unchanged_values.notna().any() else np.nan,
            'changed_minus_unchanged': float(changed_values.mean() - unchanged_values.mean()) if changed_values.notna().any() and unchanged_values.notna().any() else np.nan,
        })
feature_contrast = pd.DataFrame(contrast)

out.to_csv(SIDE_CARS['target_contract'], index=False)
adjusted_rows.to_csv(SIDE_CARS['adjusted_rows'], index=False)
scope_audit.to_csv(SIDE_CARS['scope_audit'], index=False)
target_distribution.to_csv(SIDE_CARS['distribution'], index=False)
bucket_delta.to_csv(SIDE_CARS['bucket_delta'], index=False)
feature_contrast.to_csv(SIDE_CARS['feature_contrast'], index=False)
consumed_audit.to_csv(SIDE_CARS['consumed_audit'], index=False)

target_contract_gate = bool(scope_audit['target_contract_gate'].all())
wf2022_unchanged = bool(scope_audit.loc[scope_audit['fold'].eq('wf_2022'), 'changed_rows_all'].iloc[0] == 0)
wf2020_cap_rows = int(adjusted_rows['drawdown_upside_adjustment_reason'].eq(cap['reason']).sum())
wf2024_upside_rows = int(adjusted_rows['drawdown_upside_adjustment_reason'].eq(floor['reason']).sum())
mismatch_rows = int(consumed_audit['mismatch_rows'].sum())
changed_eval_rows = int(scope_audit['changed_rows_valid'].sum() + scope_audit['changed_rows_test'].sum())

assert target_contract_gate, scope_audit
assert wf2022_unchanged, scope_audit
assert wf2020_cap_rows > 0, wf2020_cap_rows
assert wf2024_upside_rows > 0, wf2024_upside_rows
assert mismatch_rows == 0, consumed_audit
assert changed_eval_rows == 0, scope_audit

display(scope_audit)
display(consumed_audit)
display(target_distribution)


,fold,target_fold,all_target_rows,train_target_rows,valid_overlap_rows,test_overlap_rows,changed_rows_all,changed_rows_train,changed_rows_valid,changed_rows_test,target_contract_gate
0,wf_2020,wf_2020_train,2470,2218,252,0,92,92,0,0,True
1,wf_2022,wf_2022_train,505,253,252,0,0,0,0,0,True
2,wf_2023_2024,wf_2023_2024_train,756,756,0,0,151,151,0,0,True


,fold,rows,adjusted_rows,mismatch_rows,state_flag_changed_rows,mean_source_consumed_target,mean_drawdown_upside_consumed_target,mean_delta_vs_source_consumed,sum_positive_delta,sum_negative_delta
0,wf_2020_train,2470,92,0,92,0.338134,0.326960,-0.011174,0.000000,-27.6
1,wf_2022_train,505,0,0,0,0.198093,0.198093,0.000000,0.000000,0.0
2,wf_2023_2024_train,756,151,0,151,0.214621,0.253613,0.038992,29.478179,0.0


,version,fold,rows,mean_target_position,median_target_position,share_ge_35pct,share_ge_50pct,share_ge_55pct,share_ge_65pct,rows_ge_35pct,rows_ge_50pct,rows_ge_55pct,rows_ge_65pct,downtrend_rows,rebound_rows,drawdown_control_rows
0,source_consumed,wf_2020_train,2470,0.338134,0.250000,0.370445,0.272874,0.208097,0.191093,915,674,514,472,873,42,873
1,source_consumed,wf_2022_train,505,0.198093,0.146277,0.118812,0.067327,0.057426,0.043564,60,34,29,22,234,13,0
2,source_consumed,wf_2023_2024_train,756,0.214621,0.122797,0.260582,0.238095,0.043651,0.034392,197,180,33,26,478,165,0
3,drawdown_upside,wf_2020_train,2470,0.326960,0.250000,0.370445,0.235628,0.170850,0.153846,915,582,422,380,873,42,873
4,drawdown_upside,wf_2022_train,505,0.198093,0.146277,0.118812,0.067327,0.057426,0.043564,60,34,29,22,234,13,0
5,drawdown_upside,wf_2023_2024_train,756,0.253613,0.125181,0.342593,0.324074,0.243386,0.034392,259,245,184,26,478,165,0


In [5]:

pivot = target_distribution.pivot_table(
    index='fold', columns='version', values=['mean_target_position', 'share_ge_35pct', 'share_ge_55pct', 'share_ge_65pct'], aggfunc='first'
)
pivot.columns = [f'{metric}_{version}' for metric, version in pivot.columns]
pivot = pivot.reset_index()
for metric in ['mean_target_position', 'share_ge_35pct', 'share_ge_55pct', 'share_ge_65pct']:
    src = f'{metric}_source_consumed'
    dst = f'{metric}_drawdown_upside'
    if src in pivot.columns and dst in pivot.columns:
        pivot[f'{metric}_delta'] = pivot[dst] - pivot[src]

readiness_checks = {
    'target_contract_gate': target_contract_gate,
    'wf_2020_drawdown_cap_rows_gt0': wf2020_cap_rows > 0,
    'wf_2023_2024_upside_rows_gt0': wf2024_upside_rows > 0,
    'wf_2022_guard_unchanged': wf2022_unchanged,
    'changed_eval_rows_zero': changed_eval_rows == 0,
    'consumed_target_mismatch_zero': mismatch_rows == 0,
}
readiness_gate = bool(all(readiness_checks.values()))
decision = 'release16_rulebase_reversal_drawdown_upside_target_ready_for_checkpoint_replay_no_promotion' if readiness_gate else 'release16_rulebase_reversal_drawdown_upside_target_blocked_no_promotion'
next_mechanism = 'rerun_rulebase_reversal_drawdown_upside_checkpoint_metric_replay' if readiness_gate else 'repair_drawdown_upside_target_contract_before_replay'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': DIRECT_COMPARISON_BASELINE,
    'source_card': SOURCE_CARD_NAME,
    'source_target_contract': rel(SOURCE_CONTRACT),
    'source_decision': source_summary.get('decision'),
    'source_failed_fold_diagnostic': 'release16_rulebase_reversal_failed_fold_diagnostic_v1',
    'source_failed_fold_diagnostic_decision': diagnostic_summary.get('decision'),
    'reference_baseline': REFERENCE_BASELINE,
    'role': 'diagnose',
    'changed_layer': ['training_target_composition', 'model_selection', 'bc_checkpoint_target_contract'],
    'target_window': list(FAILED_FOLDS),
    'guard_folds': list(GUARD_FOLDS),
    'accepted_tradeoff': 'Reduce wf_2020 high-drawdown rulebase exposure while increasing wf_2023_2024 train-only upside participation labels; preserve wf_2022 guard fold unchanged.',
    'hypothesis': 'The prior target crossed the drawdown ceiling in wf_2020 and underparticipated in wf_2023_2024, so label semantics need both a drawdown cap and an upside floor before another official replay.',
    'acceptance_criteria': [
        'all required folds have train target rows and no valid/test target overlap',
        'wf_2022 guard-fold target rows are unchanged',
        'wf_2020 receives drawdown cap rows consumed as risk targets',
        'wf_2023_2024 receives upside floor rows consumed as rebound targets',
        'no actor promotion or retention occurs from this card',
    ],
    'failure_mode': 'If the next official replay still fails, further target relabeling is likely insufficient and the next lever should be model/objective architecture or checkpoint selector constraints.',
    'closest_historical_success_case': 'release16_rulebase_reversal_checkpoint_metric_replay_v1 lifted returns over the prior release16 target and passed wf_2022, showing the rulebase feature family can transfer under guard folds.',
    'closest_historical_failure_case': 'release16_rulebase_reversal_failed_fold_diagnostic_v1 found wf_2020 drawdown overexposure and wf_2023_2024 upside underparticipation after the rulebase target.',
    'why_not_duplicate': 'This card changes consumed target states, not only threshold names: wf_2020 labels are projected to risk/downtrend caps and wf_2023_2024 labels are projected to rebound/upside floors.',
    'recipe': RECIPE,
    'target_contract_gate': target_contract_gate,
    'target_revision_ready': readiness_gate,
    'next_training_allowed_after_target_revision': readiness_gate,
    'checkpoint_replay_allowed_next': readiness_gate,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'readiness_checks': readiness_checks,
    'wf2020_drawdown_cap_rows': wf2020_cap_rows,
    'wf2023_2024_upside_rows': wf2024_upside_rows,
    'wf2022_changed_rows': int(scope_audit.loc[scope_audit['fold'].eq('wf_2022'), 'changed_rows_all'].iloc[0]),
    'changed_eval_rows': changed_eval_rows,
    'consumed_target_mismatch_rows': mismatch_rows,
    'distribution_summary': pivot.to_dict('records'),
    'target_contract_path': rel(SIDE_CARS['target_contract']),
    'next_mechanism': next_mechanism,
    'primary_blocker': 'target_revision_ready' if readiness_gate else 'target_revision_blocked',
    'sidecars': {name: rel(path) for name, path in SIDE_CARS.items()},
}
summary = dict(selection)
summary['source_rows'] = int(len(source))
summary['adjusted_rows'] = int(len(adjusted_rows))
summary['scope_audit'] = scope_audit.to_dict('records')
summary['consumed_audit'] = consumed_audit.to_dict('records')
summary['conclusion'] = [
    f"wf_2020 drawdown cap rows: {wf2020_cap_rows}; cap projects source consumed target to {RECIPE['wf_2020_drawdown_cap']['cap_position']:.2f} on high-drawdown train states.",
    f"wf_2023_2024 upside floor rows: {wf2024_upside_rows}; floor projects source consumed target to {RECIPE['wf_2023_2024_upside_floor']['floor_position']:.2f} on train-only rebound states.",
    f"wf_2022 changed rows: {selection['wf2022_changed_rows']}; changed eval/test rows: {changed_eval_rows}; consumed-target mismatches: {mismatch_rows}.",
    'No actor is promoted; official checkpoint replay is required next.',
]
write_json(SIDE_CARS['selection'], selection)
write_json(SIDE_CARS['summary'], summary)

lines = [
    '# Release16 Rulebase Reversal Drawdown/Upside Target',
    '',
    f"Decision: `{decision}`.",
    '',
    f"Source target: `{selection['source_target_contract']}` from `{SOURCE_CARD_NAME}`.",
    f"Reference baseline: `{REFERENCE_BASELINE}`.",
    '',
    'Target changes:',
    f"- wf_2020: `{wf2020_cap_rows}` high-drawdown rulebase-reversal train rows projected to risk/downtrend target `{RECIPE['wf_2020_drawdown_cap']['cap_position']:.2f}`.",
    f"- wf_2023_2024: `{wf2024_upside_rows}` train-only upside rows projected to rebound target floor `{RECIPE['wf_2023_2024_upside_floor']['floor_position']:.2f}`.",
    f"- wf_2022 guard fold changed rows: `{selection['wf2022_changed_rows']}`.",
    '',
    f"Scope/mismatch checks: changed eval/test rows `{changed_eval_rows}`, consumed-target mismatches `{mismatch_rows}`.",
    '',
    'Risk note: this card writes only a target contract. It does not train, promote, or retain an actor, and it does not relax the walk-forward gates.',
    '',
    f"Next action: `{next_mechanism}`.",
    '',
    'Artifacts: `drawdown_upside_bc_target_contract.csv`, `drawdown_upside_adjusted_rows.csv`, `target_scope_audit.csv`, `target_distribution_before_after.csv`, `target_bucket_delta.csv`, `target_feature_contrast.csv`, `consumed_target_audit.csv`, `selection.json`, `summary.json`.',
]
SIDE_CARS['conclusion'].write_text('\n'.join(lines).rstrip() + '\n', encoding='utf-8')
display(Markdown(SIDE_CARS['conclusion'].read_text(encoding='utf-8')))


# Release16 Rulebase Reversal Drawdown/Upside Target

Decision: `release16_rulebase_reversal_drawdown_upside_target_ready_for_checkpoint_replay_no_promotion`.

Source target: `runtime/model_selection/card/release16_rulebase_reversal_checkpoint_readiness_v1/rulebase_reversal_bc_target_contract.csv` from `release16_rulebase_reversal_checkpoint_readiness_v1`.
Reference baseline: `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02`.

Target changes:
- wf_2020: `92` high-drawdown rulebase-reversal train rows projected to risk/downtrend target `0.35`.
- wf_2023_2024: `151` train-only upside rows projected to rebound target floor `0.55`.
- wf_2022 guard fold changed rows: `0`.

Scope/mismatch checks: changed eval/test rows `0`, consumed-target mismatches `0`.

Risk note: this card writes only a target contract. It does not train, promote, or retain an actor, and it does not relax the walk-forward gates.

Next action: `rerun_rulebase_reversal_drawdown_upside_checkpoint_metric_replay`.

Artifacts: `drawdown_upside_bc_target_contract.csv`, `drawdown_upside_adjusted_rows.csv`, `target_scope_audit.csv`, `target_distribution_before_after.csv`, `target_bucket_delta.csv`, `target_feature_contrast.csv`, `consumed_target_audit.csv`, `selection.json`, `summary.json`.
